# Imports

In [1]:
import pandas as pd
import pgeocode

# Data

In [3]:
df = pd.read_parquet('./data/pp-complete.parquet')
df.head()

,price,date,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type
0,166500,1995-11-22,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A
1,59000,1995-09-27,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A
2,118000,1995-12-15,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A
3,48500,1995-01-27,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A
4,27500,1995-04-20,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A


In [4]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
first_date = df['date'].min()
df['passed'] = (df['date'] - first_date).dt.days

df.drop(columns=['date'], inplace=True)

df.head()

,price,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type,year,month,passed
0,166500,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A,1995,11,325
1,59000,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A,1995,9,269
2,118000,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A,1995,12,348
3,48500,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A,1995,1,26
4,27500,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A,1995,4,109


# Feature Engineering

In [ ]:
district_counts = df['district'].value_counts().reset_index()
district_counts.columns = ['district', 'count']

# Получаем список названий district, где число упоминаний < 100
low_count_districts = district_counts[district_counts['count'] < 100]['district'].tolist()

# Удаляем эти районы из основного датафрейма
df = df[~df['district'].isin(low_count_districts)]

df['median_price_all_type'] = df.groupby(['type', 'old_new', 'duration', 'ppd_type', 'district', 'year', 'month'])['price'].transform('median')
df.head(10)

,price,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type,year,month,passed,median_price_all_type
0,166500,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A,1995,11,325,140000.0
1,59000,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A,1995,9,269,55000.0
2,118000,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A,1995,12,348,106000.0
3,48500,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A,1995,1,26,48000.0
4,27500,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A,1995,4,109,43000.0
5,114000,SE24 0DH,S,N,F,13,BRANTWOOD ROAD,LONDON,LONDON,LAMBETH,GREATER LONDON,A,1995,6,180,95750.0
6,46000,TS10 2DJ,S,N,F,2,MALCOLM GROVE,REDCAR,REDCAR,LANGBAURGH-ON-TEES,CLEVELAND,A,1995,9,262,44500.0
7,52500,SN2 2SY,S,N,F,11,MARIGOLD CLOSE,SWINDON,SWINDON,THAMESDOWN,THAMESDOWN,A,1995,5,131,55250.0
8,19000,PO32 6EP,T,N,F,17,CLARENCE ROAD,EAST COWES,EAST COWES,ISLE OF WIGHT,ISLE OF WIGHT,A,1995,10,302,37750.0
9,67500,SE18 2DH,T,N,F,75,HIGHMEAD,LONDON,LONDON,GREENWICH,GREATER LONDON,A,1995,11,320,55000.0


In [ ]:
nomi = pgeocode.Nominatim("gb")

# запрос сразу по вектору — вернёт DataFrame с полями latitude/longitude и прочими
geo = nomi.query_postal_code(df["postcode"].tolist())

df["latitude"]  = geo["latitude"].values
df["longitude"] = geo["longitude"].values

df.drop(columns=['postcode'], inplace=True)
df.dropna(subset=["latitude", "longitude"], inplace=True)

df.head()

,price,date,postcode,type,old_new,duration,PAON SAON,street,locality,town_city,district,county,ppd_type,latitude,longitude
0,166500,1995-11-22,CM23 4PA,D,Y,F,19,MAYFLOWER GARDENS,BISHOP'S STORTFORD,BISHOP'S STORTFORD,EAST HERTFORDSHIRE,HERTFORDSHIRE,A,51.895390,0.156180
1,59000,1995-09-27,L12 0AY,D,N,L,7,TRENT CLOSE,LIVERPOOL,LIVERPOOL,LIVERPOOL,MERSEYSIDE,A,53.432800,-2.909600
2,118000,1995-12-15,SL3 8XX,D,N,F,24,SOUTHWOLD SPUR,SLOUGH,SLOUGH,SLOUGH,SLOUGH,A,51.502114,-0.551257
3,48500,1995-01-27,CV12 8TF,S,N,F,5,CHELTENHAM CLOSE,BEDWORTH,BEDWORTH,NUNEATON AND BEDWORTH,WARWICKSHIRE,A,52.478075,-1.447600
4,27500,1995-04-20,SY11 1HP,S,N,F,105,BEATRICE STREET,OSWESTRY,OSWESTRY,OSWESTRY,SHROPSHIRE,A,52.832861,-2.965928
